# 5 · Authoring a measure in the notebook
Notebook 3 ran a **published** measure: two `.cql` files in a folder, resolved by name, with nothing
authored on the page. This is the other half — **drafting** one. The measure below exists only in
this notebook: a cell of shared logic about patient visits, and a second cell that `include`s it to
define an initial population.

That is how a measure is actually written. You do not start with a content repo; you start with a
question, split the parts that other measures will want to share into their own library, and publish
once the numbers hold up. Splitting logic across cells does not mean publishing it to a folder first,
which would be a strange thing to have to do to a draft.

**A `%%cql` cell that declares a `library` header is a library.** A later cell can `include` it by
name, exactly as it would include something out of a Git Folder. The last section publishes the draft
to a folder and runs it from there unchanged, which is the whole point of drafting this way.

A cell *without* a header joins the notebook's session library instead (notebook 1, section 8). That
is for exploring. A header is what makes logic a named library you can include, version and publish.

**Runs on serverless — no cluster to attach.** Just **Run All**.

In [0]:
%reload_ext cql_databricks.compiler
from cql_databricks.compiler import cql

In [0]:
# Start clean, so Run All gives the same output however often it runs: drop any earlier mount and any
# library a `%%cql` cell declared (CLIENT-SESSION-RESET). This notebook redefines a library in place, so
# a stale draft from an earlier run would otherwise answer the first include.
from cql_databricks.compiler import reset

reset(spark)

## 1. The data beside this notebook
Just one directory — **no `measures/`** — that is the difference from notebook 3. The logic comes
from the cells below; only the data is read from disk:

```
data/         FHIR NDJSON, one file per resource type
```

It is the same 4,000-patient dataset notebook 3 uses.

**Expected output:** the resolved folder, then the data directory — for example
```
content : /Workspace/Shared/CqlDatabricks/examples
found   : ['data']
```

In [0]:
import pathlib

_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
CONTENT = pathlib.Path("/Workspace") / pathlib.PurePosixPath(_nb).parent.relative_to("/")

# Fail loud with the path rather than later with a confusing empty result.
missing = [d for d in ("data",) if not (CONTENT / d).is_dir()]
if missing:
    raise FileNotFoundError(
        f"{missing} not found beside this notebook in {CONTENT} — they are published alongside it; if you "
        f"copied the notebook on its own, copy them too, or point CONTENT at your own content directory")

print("content :", CONTENT)
print("found   :", sorted(d for d in ("data",)))

## 2. The data
Unchanged from notebook 3, and deliberately so: every `[Resource]` retrieve reads the environment's view of
its type. Authoring the logic in cells changes nothing about how data reaches it — that is the seam holding
still while everything above it moves.

**Expected output:** the environment's name and catalog, then one line per type: its row count and the view
it is read from.

In [0]:
from cql_databricks.resolvers import EnvironmentResolver
from cql_databricks.data_bindings import spark_runner

environment = EnvironmentResolver(spark_runner(spark)).default()
print("environment:", environment.name, "in catalog", environment.catalog)
for t in ("Condition", "Encounter", "Patient"):
    view = f"{environment.schema('data')}.{t.lower()}"
    print(f"{t:12} {spark.table(view).count()} rows -> {view}")

## 3. Connect — with nothing mounted as a library source
No `library_path` this time. There is no folder of CQL to point at, because the libraries are about
to be written in the next two cells.

**Expected output:** `libraries declared so far: []` — an empty session, which is the honest state
of a notebook before anything has been authored.

In [0]:
from cql_databricks.compiler import connect

client = connect(spark)
print("libraries declared so far:", [str(r) for r in client.library_provider.declared()])

## 4. The shared library — a cell declares, it does not run
A helper library first: what does it mean for a patient to have visits during the measurement period?
It goes in its own library because "had an encounter" is not specific to one measure — the next measure
you write wants the same definition to mean the same thing.

A `%%cql` cell is a definition, not a question, like a Python `def`. Running it checks the CQL and
declares the library; it reads no data. A Python cell runs a define when you want its result.

**Expected output:** `declared NotebookVisits|1.0.0 — include it, or run it from Python with cql(library="NotebookVisits")`

In [0]:
%%cql
library NotebookVisits version '1.0.0'

using FHIR version '4.0.1'

include FHIRHelpers version '4.0.1' called FHIRHelpers

parameter "Measurement Period" Interval<DateTime>
  default Interval[@2026-01-01T00:00:00.000, @2026-12-31T23:59:59.999]

context Patient

define "Has Visit During Period":
  exists ([Encounter] E
    where E.period during "Measurement Period")

## 5. The measure — a second cell that `include`s the first
`include NotebookVisits` resolves against **this session**. The cell above is the library it
finds, and its text ships with the compile as a dependency, exactly as a file out of a Git Folder
would. Nothing was written to disk in between.

Like the first cell, it declares and runs nothing. It defines `Initial Population` and other checks,
which run when section 6 calls `exec()`.

**Expected output:** `declared NotebookMeasure|1.0.0 — include it, or run it from Python with cql(library="NotebookMeasure")`

In [0]:
%%cql
library NotebookMeasure version '1.0.0'

using FHIR version '4.0.1'

include FHIRHelpers version '4.0.1' called FHIRHelpers
include NotebookVisits version '1.0.0' called Visits

parameter "Measurement Period" Interval<DateTime>
  default Interval[@2026-01-01T00:00:00.000, @2026-12-31T23:59:59.999]

context Patient

define "Has Visit":
  Visits."Has Visit During Period"

define "Has Any Condition":
  exists ([Condition])

define "Initial Population":
  "Has Visit" and "Has Any Condition"

## 6. The draft is addressable by name
Both cells are libraries the session can list and bind, so the Python API reaches them the same way
it reaches published content. Python is where a cell's defines run, and where the measurement
period is bound to a different year.

Watch the population turn over: in 2025 no patient has both a visit and a condition. The patients seen
in 2025 have no condition on record. The measure that reports 1,223 of 4,000 for 2026 reports 0 of 4,000
for 2025 — the same content, honestly producing both numbers.

**One thing to know before you draft a second measure in the same notebook.** Every define is
materialized as a session temp view named after it, so two measures that both define `Initial Population`
overwrite each other's view and the older frame quietly starts returning the newer measure's rows.
Set `spark.conf.set("spark.cqldatabricks.objectPrefix", "auto")` before connecting to keep each
library's objects apart — worth doing the moment a session holds more than one measure, which is
exactly what authoring in a notebook invites.

**Expected output:**
```
declared: ['NotebookVisits|1.0.0', 'NotebookMeasure|1.0.0']
2026: initial_population=27
2025: initial_population=0
```

In [0]:
from cql_databricks.compiler import VALUE_COLUMN

print("declared:", [str(r) for r in client.libraries()])

measure = client.cql(library="NotebookMeasure")

for year in (2026, 2025):
    pops = measure.with_measurement_period(f"{year}-01-01", f"{year}-12-31").exec()
    initial_pop = pops["Initial Population"].collect()
    count = sum(1 for r in initial_pop if r[VALUE_COLUMN])
    print(f"{year}: initial_population={count}")

## 7. Editing a definition
Re-running a cell **replaces** the library it declared. Make a simple edit to the visits library and
the measure that includes it answers differently on the next run — no re-import, no restart, no publish step.

That is also the one hazard worth naming out loud: **the session is the source of truth here, not the
page.** A cell you edited but never re-ran is not the definition the next compile sees. If a number
ever surprises you mid-draft, re-run the library cell before believing it.

The edit below widens the visits check to count a visit up to a year after the period ends. Run for 2025,
the measure counted no patients in section 6. With the edit it also counts the 2026 visits, and every
patient with a condition had one.

**Expected output:** `widened_2025: initial_population=27`

In [0]:
# The same library, redeclared from Python rather than a cell: `register_cell` is what `%%cql` calls, so a
# generated or parameterized variant of a definition is a first-class way to declare one.
from pyspark.sql import functions as F

from cql_databricks.compiler import register_cell

draft = client.library_provider.source("NotebookVisits").cql
# Edit: count a visit up to a year after the period ends. Fail loud if the text to edit is not there.
ORIGINAL = 'E.period during "Measurement Period"'
WIDENED = 'E.period during Interval[start of "Measurement Period", end of "Measurement Period" + 1 year]'
if ORIGINAL not in draft:
    raise ValueError(f"NotebookVisits no longer contains {ORIGINAL!r}; update this edit")
register_cell(draft.replace(ORIGINAL, WIDENED))

edited_measure = client.cql(library="NotebookMeasure").with_measurement_period("2025-01-01", "2025-12-31")
initial_pop = edited_measure.exec("Initial Population").collect()
count = sum(1 for r in initial_pop if r[VALUE_COLUMN])
print("widened_2025: initial_population=", count, sep="")

## 8. Publishing the draft
A published library is the same text in a file. Writing the two cells out and mounting that directory
gives a folder like the one notebook 3 runs from — and the measure resolves by name from it with no
change to a single line of CQL.

In a real workflow the directory is your measures repo as a **Databricks Git Folder**, and publishing
is a commit. The point here is only that nothing about the draft was notebook-shaped: the cell was
always the file.

**You publish the draft you have.** The previous section left `NotebookVisits` sitting with the
widened-window edit, and writing the session out now would publish exactly that — so the experiment
is put back first. Worth doing deliberately rather than discovering it in a file: the notebook holds
one current definition per name, not a history you can pick a version out of.

Cells still resolve ahead of the folder, so `clear()` is what hands the names back to the published
copies once both exist.

The reverse never happens. A measure resolved **from the folder** has its whole `include` closure
resolved there too, so a same-named cell cannot change what a published measure means — overriding is
something you opt into by writing the measure as a cell, never something that happens to a measure you
merely ran. A notebook that could quietly redefine published content would make every number out of it
impossible to cite.

**Expected output:**
```
published: ['NotebookMeasure-1.0.0.cql', 'NotebookVisits-1.0.0.cql']
from the folder: initial_population=27
```

In [0]:
import tempfile

# Put the original NotebookVisits definition back; the widened window was an experiment.
register_cell(client.library_provider.source("NotebookVisits").cql.replace(WIDENED, ORIGINAL))

published = pathlib.Path(tempfile.mkdtemp(prefix="cql_databricks-draft-"))
for ref in client.library_provider.declared():
    (published / f"{ref.name}-{ref.version}.cql").write_text(
        client.library_provider.source(ref.name).cql)
print("published:", sorted(p.name for p in published.glob("*.cql")))

# Mount the folder, then drop the drafts so the published copies are what the names resolve to.
from_folder = connect(spark, library_path=str(published))
from_folder.library_provider.clear()

pops = from_folder.cql(library="NotebookMeasure").exec("Initial Population")
print(f"from the folder: initial_population={pops.filter(F.col(VALUE_COLUMN)).count()}")

---
### What just happened
A helper library written in one cell, a measure that includes it in a second cell, both run over FHIR data,
re-parameterized to a second year, one of them edited in place and re-run to see the change, and finally
both written to a folder where they run by name unchanged. No registry, no application, and no publish
step until the last section — which is the order you actually want: publish what already works.

**03 · A clinical quality measure** is the same shape from the other end: content that already exists,
run from a folder with nothing authored on the page.

In [0]:
import json
dbutils.notebook.exit(json.dumps({"ok": True, "notebook": "05_authoring_in_the_notebook"}))